In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# ==============================
# Load Model and Face Detector
# ==============================
model = load_model("mask_detector.model")

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Labels and Colors
labels_dict = {0: "No Mask", 1: "Mask"}
color_dict = {0: (0, 0, 255), 1: (0, 255, 0)}

# ==============================
# Start Webcam
# ==============================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot access webcam")
    exit()

print("Press ESC to exit")

# ==============================
# Real-Time Detection Loop
# ==============================
while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to grab frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(30, 30)
    )

    for (x, y, w, h) in faces:
        face_img = frame[y:y+h, x:x+w]

        try:
            resized = cv2.resize(face_img, (224, 224))
        except:
            continue

        normalized = resized / 255.0
        reshaped = np.reshape(normalized, (1, 224, 224, 3))

        result = model.predict(reshaped, verbose=0)
        label = int(result[0][0] > 0.5)  # Binary classification

        color = color_dict[label]
        text = labels_dict[label]

        # Draw rectangle
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

        # Put label
        cv2.putText(frame, text,
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2)

    cv2.imshow("Real-Time Face Mask Detection", frame)

    key = cv2.waitKey(1)
    if key == 27:  # ESC key
        break

# ==============================
# Cleanup
# ==============================
cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'cv2'